How does the application know Alice is really Alice before it gives her identity to the agent?

## Create a synthetic token

A JWT generally consists of three encoded sections:

`Header.Payload.Signature`

The payload contains claims such as:
- sub: Who the token represents.
- iss: Who issued it.
- aud: Which application it is intended for.
- exp: When it expires.
The signature lets the verifier detect unauthorized modifications and validate the trusted issuer's signing authority.

For this notebook, we'll simulate the identity provider with an ephemeral symmetric key generated in memory. This is a teaching fixture, not our production authentication design.

In [1]:
import secrets
from datetime import datetime, timedelta, timezone

import jwt

# Ephemeral lab-only signing key
SECRET = secrets.token_bytes(32)

ISSUER = "investigation-lab"
AUDIENCE = "investigation-api"

now = datetime.now(timezone.utc)

alice_token = jwt.encode(
    {
        "sub": "E-101",
        "iss": ISSUER,
        "aud": AUDIENCE,
        "iat": now,
        "exp": now + timedelta(minutes=15),
    },
    SECRET,
    algorithm="HS256",
)

print("Synthetic JWT created")

Synthetic JWT created


In [2]:
# alice_token

## Verify before trusting

In [3]:
def verify_token(token: str) -> dict:
    return jwt.decode(
        token,
        SECRET,
        algorithms=["HS256"],
        issuer=ISSUER,
        audience=AUDIENCE,
        options={
            "require": ["sub", "iss", "aud", "exp", "iat"]
        },
    )


verified_claims = verify_token(alice_token)

print("Authenticated subject:", verified_claims["sub"])

Authenticated subject: E-101


## Test security failures - Run controlled tests

In [13]:
invalid_tokens = {
    "Tampered signature": str(alice_token)[:-1] + (
        "A" if alice_token[-1] != "A" else "B"
    ),
    "Wrong audience": jwt.encode(
        {
            "sub": "E-101",
            "iss": ISSUER,
            "aud": "another-application",
            "exp": now + timedelta(minutes=15),
            "iat": now,
        },
        SECRET,
        algorithm="HS256",
    ),
    "Expired token": jwt.encode(
        {
            "sub": "E-101",
            "iss": ISSUER,
            "aud": AUDIENCE,
            "exp": now - timedelta(minutes=1),
            "iat": now - timedelta(minutes=20),
        },
        SECRET,
        algorithm="HS256",
    ),
}

for name, token in invalid_tokens.items():
    try:
        verify_token(token)
        print(f"{name}: ACCEPTED")
    except jwt.InvalidTokenError:
        print(f"{name}: REJECTED")

Tampered signature: REJECTED
Wrong audience: REJECTED
Expired token: REJECTED


## Trusted Employee Directory

What if Alice leaves the company, but the token is still valid?

Should the API still allow Alice to investigate claims because her JWT is cryptographically valid? What additional control would you introduce, and where would it live?

The solution is to check both token validity and current account eligibility.

```
Incoming JWT
     |
     v
Verify signature, issuer, audience, expiry
     |
     v
Resolve employee in trusted directory
     |
     +-- Employee missing? --> DENY
     |
     +-- Employee disabled? --> DENY
     |
     v
Construct trusted Principal
     |
     v
LangGraph runtime context
     |
     v
Authorization gateway
     |
     v
Allowed claim data
```

In [15]:
employees = {
    "E-101": {"tenant_id": "northstar", "active": True},
    "E-102": {"tenant_id": "northstar", "active": True},
}

def resolve_active_employee(verified_claims):
    employee_id = verified_claims["sub"]

    employee = employees.get(employee_id)

    if not employee or not employee["active"]:
        raise PermissionError("Employee not authorized")

    return {
        "employee_id": employee_id,
        "tenant_id": employee["tenant_id"],
    }

In [16]:
try: 
    verified_claims = verify_token(alice_token)
    print(resolve_active_employee(verified_claims))
except (jwt.InvalidTokenError, PermissionError) as e:
    print("Error:", e)  

{'employee_id': 'E-101', 'tenant_id': 'northstar'}


### Disable Alice without changing her token

In [17]:
employees["E-101"]["active"] = False

# The existing token is still cryptographically valid.
verified_claims = verify_token(alice_token)

print("JWT verification succeeded")

# But the application should now reject Alice.
try:
    resolve_active_employee(verified_claims)
except PermissionError:
    print("Application access denied: employee disabled")

JWT verification succeeded
Application access denied: employee disabled


A valid token answers:

"Was this credential issued by a trusted authority, for this application, and is it still valid under our token-validation rules?"

An active-directory/employee-eligibility check answers:

"Is this employee currently allowed to use our investigation application?"

Resource authorization answers:

"Can this employee access this particular claim?"

All three checks matter.